# Notebook 07 — EfficientNetV2 Architecture Deep Dive

In this notebook we open the hood of **EfficientNetV2** and walk through every
design choice that makes it both accurate and efficient. There is **no training
here** — this is pure architectural understanding. By the end you will be able
to read the model's layer list and know exactly what each block does and why.

## Learning goals

- Place EfficientNetV2 in the historical lineage (MobileNet ➜ V1 ➜ V2).
- Explain **compound scaling** along depth, width, and resolution.
- Understand the **MBConv** (inverted-bottleneck) block and why it is parameter-efficient.
- Show that **depthwise separable convolution** has ~8x fewer parameters than a standard 3x3 conv.
- Recognise a **Squeeze-and-Excitation (SE)** sub-module inside MBConv.
- Understand V2's new **Fused-MBConv** block and why it is faster in early stages.
- See **stochastic depth / DropPath** in action.
- Inspect the full V2-S model with `torchinfo.summary` and group it into stages.
- Visualise intermediate feature maps to see what each stage learns.
- Know what **progressive learning** means (the training recipe used in notebook 12).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/numberonewastefellow/my_learning/blob/main/notebooks/07_efficientnetv2_architecture.ipynb)

In [ ]:
# Universal setup: runs identically in Colab and locally
import sys, os
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("my_learning"):
        !git clone --quiet https://github.com/numberonewastefellow/my_learning.git
    %cd my_learning
    !pip install -q -r requirements.txt

repo_root = os.path.abspath(".") if IN_COLAB else os.path.abspath("..")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from utils.env import bootstrap
info = bootstrap()
device = info.device

## 1. Historical context — a very short tour

| Year | Model | Key idea |
|------|-------|----------|
| 2017 | MobileNetV1 | **Depthwise separable** convolutions for mobile. |
| 2018 | MobileNetV2 | Inverted **residual** block with linear bottleneck (the ancestor of MBConv). |
| 2018 | SENet       | **Squeeze-and-Excitation** channel attention. |
| 2019 | EfficientNet **V1** | **Compound scaling** of depth/width/resolution; MBConv backbone. |
| 2021 | EfficientNet **V2** | **Fused-MBConv** for early stages; smaller models, faster training, progressive learning. |

Selected ImageNet accuracy (top-1) vs parameter count:

| Model | Params | Top-1 | Input res |
|-------|--------|-------|-----------|
| V1-B0 | 5.3 M  | 77.3% | 224 |
| V1-B3 | 12 M   | 81.6% | 300 |
| V2-S  | 21.5 M | 83.9% | 300-384 |
| V2-M  | 54 M   | 85.1% | 384-480 |
| V2-L  | 119 M  | 85.7% | 384-480 |

**Takeaway.** V2 is a *refinement* of V1 — same compound-scaling philosophy,
but with smarter blocks in early stages and a progressive training recipe.

## 2. What is compound scaling?

Before EfficientNet, people scaled a CNN by **one** axis at a time — usually depth
(add more layers, e.g. ResNet-50 → ResNet-101). V1's insight: for a fixed compute
budget, it is better to scale **all three** axes together:

- **depth** — number of layers/blocks,
- **width** — number of channels,
- **resolution** — input image size.

The paper expresses this with three coefficients `α, β, γ` satisfying
`α · β² · γ² ≈ 2` (so that doubling `φ` roughly doubles FLOPs). For a scaling
factor `φ`, you scale:

```text
depth      = α ** φ
width      = β ** φ
resolution = γ ** φ
```

A tiny illustrative snippet below.

In [ ]:
# Conceptual illustration of compound scaling.
# Do NOT use these numbers to configure a real model — this is just the formula.
alpha, beta, gamma = 1.2, 1.1, 1.15  # values close to the V1 paper

print(f"{'phi':>4} | {'depth':>6} | {'width':>6} | {'resolution':>10}")
print("-" * 36)
for phi in range(0, 6):
    depth      = alpha ** phi
    width      = beta  ** phi
    resolution = gamma ** phi
    print(f"{phi:>4} | {depth:6.2f} | {width:6.2f} | {resolution:10.2f}")

Read the table: as `phi` grows, all three quantities grow together. The
different EfficientNet variants (B0..B7, S/M/L/XL) are just different `phi`
settings plus a few hand-tuned deviations.

## 3. The MBConv block (MobileNetV2's inverted bottleneck)

A classic ResNet block *compresses* then *expands* channels. MBConv does the
opposite — it **expands first**, does the spatial work on the expanded tensor,
then **projects back down**. Here is the pipeline with C input channels and
expansion factor `t` (typically 4 or 6):

```text
           +--------------------------------------------------------+
input  -->| 1x1 conv  ( C     -> t*C )   expand                     |
  |       | depthwise 3x3 ( t*C -> t*C )  spatial mixing              |
  |       | Squeeze-and-Excitation (channel attention)               |
  |       | 1x1 conv  ( t*C   -> C_out ) project (linear, no ReLU)  |
  |       +--------------------------------------------------------+
  +-------------------> residual add (only when stride=1 and C==C_out)
```

Key design points:

1. **Expansion** — more channels to mix means a richer intermediate representation.
2. **Depthwise** convolution — one filter per channel, cheap.
3. **Linear bottleneck** — no activation on the project; preserves the low-dim manifold.
4. **Residual** — only when shape matches; otherwise a plain stack.

Let's load a real V2-S and look at one MBConv block.

In [ ]:
import torch
import torch.nn as nn
import timm

# pretrained=False means we instantiate the architecture but skip the weight
# download. For architectural study we do not need weights.
model = timm.create_model("tf_efficientnetv2_s", pretrained=False)
model.eval()
print(type(model).__name__)
print("Number of stages in model.blocks:", len(model.blocks))

In [ ]:
# model.blocks is a Sequential of stages. Each stage is a Sequential of blocks.
# Stage 2 in V2-S uses MBConv. Let's inspect one block.
stage_idx = 2
block_idx = 0
mbconv = model.blocks[stage_idx][block_idx]
print(f"Block type at blocks[{stage_idx}][{block_idx}]: {type(mbconv).__name__}")
print()
print(mbconv)

Notice in the printout:

- `conv_pw` — the 1x1 **p**oint**w**ise expand.
- `conv_dw` — the 3x3 **d**epth**w**ise convolution (note the `groups=` equals its channels).
- `se` — a Squeeze-and-Excitation sub-module (we study it below).
- `conv_pwl` — the 1x1 pointwise project (**l**inear = no activation).
- `drop_path` — stochastic depth (we study it below).

## 4. Depthwise separable convolution — the parameter-saving trick

A **standard** 3x3 conv that maps 64 → 128 channels has:

```text
params = C_in * C_out * k * k = 64 * 128 * 9 = 73,728
```

A **depthwise-separable** version splits the job in two:

1. **Depthwise** 3x3 — one filter per input channel (`groups=C_in`). Only mixes *spatially*.
   Params: `C_in * 1 * k * k = 64 * 9 = 576`.
2. **Pointwise** 1x1 — mixes *across channels*. Params: `C_in * C_out = 64 * 128 = 8,192`.

Total: `576 + 8,192 = 8,768` — about **8.4x** fewer parameters for the same output shape.
Let's demonstrate with live code.

In [ ]:
# Count parameters of standard vs depthwise-separable 3x3 conv.
def count_params(mod: nn.Module) -> int:
    return sum(p.numel() for p in mod.parameters() if p.requires_grad)

C_in, C_out, k = 64, 128, 3

standard = nn.Conv2d(C_in, C_out, kernel_size=k, padding=1, bias=False)

# depthwise: groups == C_in, out_channels == C_in (one filter per channel)
depthwise = nn.Conv2d(C_in, C_in, kernel_size=k, padding=1,
                     groups=C_in, bias=False)
# pointwise: 1x1 conv
pointwise = nn.Conv2d(C_in, C_out, kernel_size=1, bias=False)
separable = nn.Sequential(depthwise, pointwise)

p_std = count_params(standard)
p_sep = count_params(separable)
print(f"standard 3x3          : {p_std:>8,} params")
print(f"depthwise + pointwise : {p_sep:>8,} params")
print(f"ratio                 : {p_std / p_sep:5.2f}x smaller")

# Same output shape — let's verify.
x = torch.randn(1, C_in, 32, 32)
print("standard  output:", standard(x).shape)
print("separable output:", separable(x).shape)

## 5. Squeeze-and-Excitation (SE) — channel attention

SE was introduced in **SENet (2018)** and we already met its *spirit* in
notebook 06 (`ChannelAttention`). The idea:

1. **Squeeze** — global-average-pool each channel to a single number → vector of size `C`.
2. **Excitation** — a tiny MLP (`C → C/r → C`) with sigmoid produces per-channel gates in `[0, 1]`.
3. **Scale** — multiply each channel of the feature map by its gate.

In plain English: *look at the whole image first, then decide which channels
deserve to be amplified or suppressed*. It costs only a handful of parameters
per block but consistently improves accuracy.

Let's pull the SE sub-module out of the MBConv block we inspected above.

In [ ]:
se_module = mbconv.se  # SE sub-module of the MBConv block
print(type(se_module).__name__)
print(se_module)

# SE parameter count is tiny compared to the surrounding block.
print()
print(f"SE params       : {count_params(se_module):,}")
print(f"whole MBConv    : {count_params(mbconv):,}")

## 6. Fused-MBConv — V2's key innovation

Depthwise convs are **memory-bandwidth-bound** on modern accelerators: they
touch a lot of memory per FLOP. In the **early stages** of a network the
spatial resolution is still large (e.g. 150x150) and the channel count is
small (e.g. 24) — exactly the regime where depthwise is *slower* in wall-clock
time than a plain dense conv.

V2's fix: for early stages, **fuse** `1x1 expand + 3x3 depthwise` into a single
regular `3x3 conv`. You lose a tiny bit of parameter efficiency but gain a lot
of throughput on GPUs / TPUs.

```text
MBConv        : [ 1x1 conv ] -> [ 3x3 depthwise ] -> [ SE ] -> [ 1x1 conv ]
Fused-MBConv  : [        3x3 conv        ]         -> [ SE? ] -> [ 1x1 conv ]
```

Let's verify by printing a block from stage 0 (Fused) and a block from stage 4
(MBConv) in V2-S.

In [ ]:
fused_block  = model.blocks[0][0]   # stage 0 uses Fused-MBConv in V2-S
mbconv_block = model.blocks[4][0]   # stage 4 uses MBConv

print("=== Stage 0 block (Fused-MBConv) ===")
print(type(fused_block).__name__)
print(fused_block)
print()
print("=== Stage 4 block (MBConv) ===")
print(type(mbconv_block).__name__)
print(mbconv_block)

Notice the Fused block has `conv_exp` as a plain 3x3 conv (no `groups=`) and
**no** `conv_dw`; the depthwise step has been absorbed. The later-stage MBConv
keeps the depthwise split because by then the feature maps are small and many
channels — the regime where depthwise wins.

## 7. Stochastic depth (DropPath)

For a residual block `y = x + f(x)`, **DropPath** randomly sets `f(x)` to zero
*for a whole sample in the minibatch* with probability `p`. It is like Dropout,
but at the block level.

V2 uses a **linearly increasing** drop rate: early blocks keep their signal,
later blocks drop more. This trains a deep network as an implicit ensemble of
shallower ones and improves generalisation.

In [ ]:
from timm.layers import DropPath

# 4 samples, 8 channels, 4x4 spatial. Each element is 1.0 so we can see drops.
x = torch.ones(4, 8, 4, 4)

drop = DropPath(drop_prob=0.5)
drop.train()                          # active only in train mode

torch.manual_seed(0)
y = drop(x)
# Check whether each sample was kept or dropped by summing its values.
per_sample_sum = y.sum(dim=(1, 2, 3))
print("per-sample sum (0.0 means the whole branch was dropped for that sample):")
print(per_sample_sum)

drop.eval()                            # inactive at eval — pass-through
y_eval = drop(x)
print("\nIn eval mode, DropPath is a no-op. Max diff vs input:",
      (y_eval - x).abs().max().item())

## 8. Walk through the full V2-S model

Now we zoom out to the whole network. V2-S consists of:

1. A **stem** — a single 3x3 conv that projects RGB (3 channels) to 24 channels.
2. **Stages 0..5** — groups of homogeneous blocks. Early stages use Fused-MBConv;
   later stages use MBConv.
3. A **head** — a 1x1 conv + pooling + classifier.

First, the formal summary via `torchinfo`.

In [ ]:
from torchinfo import summary

summary(
    model,
    input_size=(1, 3, 300, 300),
    depth=3,
    col_names=("input_size", "output_size", "num_params", "mult_adds"),
    verbose=1,
)

The summary above is long. Let's build a **per-stage** table so the structure
becomes obvious at a glance.

In [ ]:
import pandas as pd

rows = []
for stage_idx, stage in enumerate(model.blocks):
    # Each `stage` is a nn.Sequential of blocks.
    first = stage[0]
    block_type = type(first).__name__
    num_blocks = len(stage)

    # Figure out in/out channels and stride from the first block.
    # timm EdgeResidual (Fused) and InvertedResidual (MBConv) both expose
    # conv_pwl (project) at the end of the block.
    proj = getattr(first, "conv_pwl", None)
    if proj is None:
        proj = getattr(first, "conv_pw", None)

    # stride lives on the first spatial conv
    spatial = getattr(first, "conv_dw", None) or getattr(first, "conv_exp", None)
    stride = spatial.stride[0] if spatial is not None else "?"

    # infer channels from the block's first and last conv
    in_ch  = getattr(first, "conv_exp", getattr(first, "conv_pw", None)).in_channels
    out_ch = proj.out_channels if proj is not None else in_ch

    params = sum(p.numel() for p in stage.parameters())

    rows.append({
        "stage": stage_idx,
        "block_type": block_type,
        "num_blocks": num_blocks,
        "in_channels": in_ch,
        "out_channels": out_ch,
        "stride(1st)": stride,
        "params": f"{params:,}",
    })

df = pd.DataFrame(rows)
print(df.to_markdown(index=False))

Read the table top to bottom:

- Early stages (0, 1, 2) use `EdgeResidual` (Fused-MBConv). Small channels,
  large spatial size — fused conv wins.
- Later stages (3, 4, 5) use `InvertedResidual` (MBConv) because spatial size
  has halved a few times and channel count has grown.
- **Stride 2** blocks appear at the start of stages 1, 2, 3, 5 — these are the
  spatial-downsampling steps.
- Parameter count concentrates in the **late** stages where channels are wide.

## 9. What does each stage actually *see*?

We pass a real image through the network and snapshot the output of every
stage with forward hooks. For each stage we plot the first 16 channels of
the feature map so you can feel the progression from *edges → textures →
object parts*.

If you have no sample image, we fall back to a synthetic one.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms
from utils.env import data_dir
from utils.plotting import show_image_grid

# -------- 1. Load one sample image --------
def _find_sample_image() -> Image.Image:
    candidates = []
    root = data_dir()
    for folder in ("oxford-iiit-pet", "oxford_iiit_pet", "pets", "sample_data"):
        p = os.path.join(root, folder)
        if os.path.isdir(p):
            for r, _, files in os.walk(p):
                for f in files:
                    if f.lower().endswith((".jpg", ".jpeg", ".png")):
                        candidates.append(os.path.join(r, f))
                        break
                if candidates:
                    break
        if candidates:
            break
    if candidates:
        return Image.open(candidates[0]).convert("RGB")
    # Synthetic fallback: a simple blobby image with gradients.
    arr = np.zeros((300, 300, 3), dtype=np.uint8)
    yy, xx = np.mgrid[0:300, 0:300]
    arr[..., 0] = (xx % 64 * 4).astype(np.uint8)
    arr[..., 1] = (yy % 64 * 4).astype(np.uint8)
    arr[..., 2] = (((xx + yy) // 2) % 256).astype(np.uint8)
    return Image.fromarray(arr)

pil_img = _find_sample_image()
print("Sample image size:", pil_img.size)

# -------- 2. Preprocess --------
preprocess = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])
x = preprocess(pil_img).unsqueeze(0)   # shape (1, 3, 300, 300)
print("Input tensor shape:", x.shape)

In [ ]:
# -------- 3. Register hooks on each stage --------
activations = {}

def make_hook(name):
    def hook(mod, inp, out):
        activations[name] = out.detach()
    return hook

handles = []
for i, stage in enumerate(model.blocks):
    handles.append(stage.register_forward_hook(make_hook(f"stage_{i}")))

# -------- 4. Forward pass --------
model.eval()
with torch.no_grad():
    _ = model(x)

# -------- 5. Report shapes --------
for name, act in activations.items():
    print(f"{name}: {tuple(act.shape)}")

# Remove hooks so we don't leak memory.
for h in handles:
    h.remove()

As expected, the **spatial size halves** at each downsampling stage and the
**channel count grows**. Now let's visualise channels 0..15 of each stage.
We convert each channel to grayscale by treating it as a single-channel image.

In [ ]:
def channels_to_images(feat: torch.Tensor, n: int = 16):
    """Take feat of shape (1, C, H, W); return a list of `n` (1, H, W) tensors
    rescaled to [0, 1] for display.
    """
    feat = feat[0]  # (C, H, W)
    n = min(n, feat.shape[0])
    imgs = []
    for c in range(n):
        fm = feat[c]
        fm = (fm - fm.min()) / (fm.max() - fm.min() + 1e-6)
        # show_image_grid expects (C, H, W) tensors — use 1-channel (grayscale).
        imgs.append(fm.unsqueeze(0).cpu())
    return imgs

for name, act in activations.items():
    imgs = channels_to_images(act, n=16)
    titles = [f"ch {i}" for i in range(len(imgs))]
    print(f"\n### {name} — shape {tuple(act.shape)}")
    show_image_grid(imgs, titles=titles, ncols=4)

**What to look for.** In stage 0 you will see feature maps that fire on edges
and colour blobs. By stages 3-4 the responses are sparser and more spatially
structured — they fire on object **parts**. The last stage has only a few
tens of activations that are near-binary: *this region looks like class-k*.

## 10. Progressive learning — V2's training recipe

V2's *training-time* innovation (not an architectural one) is **progressive
learning**: start with small images + weak augmentation, then gradually grow
the image size **and** strengthen the augmentation. The rationale is that
strong augmentation on small inputs destroys signal, while weak augmentation
on big inputs does not regularise enough — so couple the two knobs.

A typical schedule for V2-S:

| Epoch range | Image size | Aug magnitude | Dropout |
|-------------|------------|---------------|---------|
|   1 – 87    | 128        | 5             | 0.10    |
|  88 – 175   | 172        | 8             | 0.20    |
| 176 – 263   | 216        | 10            | 0.30    |
| 264 – 350   | 300        | 15            | 0.40    |

We will actually *run* a progressive-learning loop in **notebook 12**. Here,
just remember that the architecture and the schedule are **co-designed**.

## Key Takeaways

- EfficientNet's backbone is a stack of **inverted-bottleneck** blocks (MBConv).
  *Expand → depthwise → SE → project*, with a residual when shapes match.
- **Depthwise separable** convolution is ~8x cheaper in parameters than a
  standard 3x3 — but it is memory-bandwidth-bound on GPUs.
- V2 solves this for early stages by **fusing** the expand + depthwise into a
  single regular 3x3 conv (**Fused-MBConv**). Late stages keep the classical
  MBConv.
- **Compound scaling** grows depth, width, and resolution together by a
  single coefficient `phi`.
- **Squeeze-and-Excitation** adds a cheap channel-attention gate inside each
  block.
- **DropPath / stochastic depth** randomises depth during training for
  implicit ensembling.
- Feature maps evolve from **edges → textures → parts → classes** as we go
  deeper into the stages.
- V2's **progressive learning** jointly schedules image size, augmentation,
  and dropout during training.

## Exercises

1. **Param budget.** Using `count_params`, compare the total parameter counts
   of `tf_efficientnetv2_s`, `tf_efficientnetv2_m`, and `tf_efficientnetv2_l`.
   How do they relate to the paper's reported numbers (21.5 / 54 / 119 M)?
2. **FLOPs vs params.** Install `fvcore` and compute FLOPs at input sizes
   192, 224, 300, 384 on V2-S. Plot FLOPs vs input resolution — is it linear
   in `H * W`? Why?
3. **Swap SE out.** Replace the SE module inside one MBConv block with
   `nn.Identity()` and verify the forward pass still runs. By how much do
   parameters drop for that block?
4. **Deeper hooks.** Hook the outputs of the *first conv inside every MBConv*
   and plot the magnitude histogram. Compare early vs late stages — where
   are activations most saturated?
5. **Fused vs MBConv speed.** Build a random `(B, 24, 150, 150)` tensor and
   benchmark one `EdgeResidual` (Fused) vs one `InvertedResidual` (MBConv)
   block of roughly the same output shape. Is the Fused version really
   faster on your hardware?